In [40]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

In [41]:
def load_and_parse(file_path):
    """Load GH Archive data and parse nested fields"""
    events = []
    import gzip, json
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            events.append(json.loads(line))
    df = pd.DataFrame(events)

    # Parse nested fields
    df['actor_login'] = df['actor'].apply(lambda x: x.get('login') if isinstance(x, dict) else None)
    df['repo_name'] = df['repo'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)
    df['minute'] = pd.to_datetime(df['created_at']).dt.minute

    return df

# Load two hours of data
df_12 = load_and_parse('raw_data/2025-01-20-12.json.gz')
df_13 = load_and_parse('raw_data/2025-01-20-13.json.gz')

print(f"Hour 12: {len(df_12)} events")
print(f"Hour 13: {len(df_13)} events")

Hour 12: 278646 events
Hour 13: 261240 events


AGGREGATE USER FEATURES


In [42]:
def aggregate_features(df, hour_label):
    
    push_events = df[df['type'] == 'PushEvent'].groupby('actor_login').size()
    pr_events = df[df['type'] == 'PullRequestEvent'].groupby('actor_login').size()
    
    aggregated = df.groupby('actor_login').agg({
        'id': 'count',
        'type': 'nunique',
        'repo_name': 'nunique',
        'minute': ['max', 'min', 'std']
    })
    
    aggregated.columns = [
        f'total_events_{hour_label}',
        f'event_type_diversity_{hour_label}',
        f'unique_repos_{hour_label}',
        f'last_active_minute_{hour_label}',
        f'first_active_minute_{hour_label}',
        f'activity_duration_{hour_label}'
    ]
    
    aggregated[f'push_count_{hour_label}'] = push_events
    aggregated[f'pr_count_{hour_label}'] = pr_events
    aggregated = aggregated.fillna(0)
    
    aggregated[f'minutes_since_last_action_{hour_label}'] = 60 - aggregated[f'last_active_minute_{hour_label}']
    
    return aggregated

X_raw = aggregate_features(df_12, 'h12')

h13_activity = df_13.groupby('actor_login').agg({'id': 'count'}).rename(columns={'id': 'total_events_h13'})
active_threshold = h13_activity['total_events_h13'].quantile(0.75)
h13_activity['is_active_next_hour'] = (h13_activity['total_events_h13'] > active_threshold).astype(int)


data = X_raw.join(h13_activity[['is_active_next_hour']], how='inner')
data['is_active_next_hour'] = data['is_active_next_hour'].fillna(0).astype(int)

print(f"Total users to predict: {len(data)}")
print(f"Class distribution:\n{data['is_active_next_hour'].value_counts(normalize=True)}")

Total users to predict: 17688
Class distribution:
is_active_next_hour
0    0.731004
1    0.268996
Name: proportion, dtype: float64


HANDLE MISSING VALUES

In [43]:
print(data.isnull().sum())

# No missing values in our aggregated features
# (actor_login with NULL were automatically excluded by groupby)

# CAP OUTLIERS (WINSORIZATION)

def cap_outliers(df, column):
    """Cap values at 99th percentile"""
    cap = df[column].quantile(0.99)
    capped_count = (df[column] > cap).sum()
    df[f'{column}_capped'] = df[column].clip(upper=cap)
    print(f"  {column}: capped {capped_count} values at {cap:.1f}")
    return df, f'{column}_capped'

# Apply capping to skewed features
data, total_events_capped = cap_outliers(data, 'total_events_h12')
data, unique_repos_capped = cap_outliers(data, 'unique_repos_h12')

total_events_h12                 0
event_type_diversity_h12         0
unique_repos_h12                 0
last_active_minute_h12           0
first_active_minute_h12          0
activity_duration_h12            0
push_count_h12                   0
pr_count_h12                     0
minutes_since_last_action_h12    0
is_active_next_hour              0
dtype: int64
  total_events_h12: capped 176 values at 45.0
  unique_repos_h12: capped 152 values at 6.0


CREATE DERIVED FEATURES

In [44]:
# 1. events_per_repo
data['events_per_repo'] = data['total_events_h12_capped'] / (data['unique_repos_h12_capped'] + 1)

# 2. diversity_score
data['diversity_score'] = data['event_type_diversity_h12'] / data['total_events_h12_capped'].clip(lower=1)

# 3. log_total_events
data['log_total_events'] = np.log1p(data['total_events_h12_capped'])

# 4. is_bot_suspect
data['is_bot_suspect'] = (data['total_events_h12_capped'] > 1000).astype(int)
bot_count = data['is_bot_suspect'].sum()


SELECT FINAL FEATURES

In [45]:
# Features to use
feature_columns = [
    'total_events_h12_capped',
    'event_type_diversity_h12',
    'unique_repos_h12_capped',
    'events_per_repo',
    'diversity_score',
    'log_total_events',
    'is_bot_suspect',
    'minutes_since_last_action_h12',
    'push_count_h12',
    'pr_count_h12'
]

X = data[feature_columns].copy()
y = data['is_active_next_hour'].copy()

print(f"Selected features: {feature_columns}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:\n{y.value_counts()}")

Selected features: ['total_events_h12_capped', 'event_type_diversity_h12', 'unique_repos_h12_capped', 'events_per_repo', 'diversity_score', 'log_total_events', 'is_bot_suspect', 'minutes_since_last_action_h12', 'push_count_h12', 'pr_count_h12']
X shape: (17688, 10)
y shape: (17688,)

Class distribution:
is_active_next_hour
0    12930
1     4758
Name: count, dtype: int64


NORMALIZATION

In [46]:
# Check skew before normalization
print("Skew before normalization:")
for col in X.columns:
    skew = X[col].skew()
    print(f"  {col}: {skew:.2f}")

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("\nStandardScaler applied (mean=0, std=1 for all features)")

# Check after normalization
print("\nStatistics after normalization:")
print(X_scaled.describe().round(2))

Skew before normalization:
  total_events_h12_capped: 4.98
  event_type_diversity_h12: 2.44
  unique_repos_h12_capped: 4.42
  events_per_repo: 5.79
  diversity_score: -0.34
  log_total_events: 1.57
  is_bot_suspect: 0.00
  minutes_since_last_action_h12: 0.82
  push_count_h12: 132.26
  pr_count_h12: 104.30

StandardScaler applied (mean=0, std=1 for all features)

Statistics after normalization:
       total_events_h12_capped  event_type_diversity_h12  \
count                 17688.00                  17688.00   
mean                      0.00                      0.00   
std                       1.00                      1.00   
min                      -0.45                     -0.51   
25%                      -0.45                     -0.51   
50%                      -0.28                     -0.51   
75%                       0.06                      0.61   
max                       7.03                     10.62   

       unique_repos_h12_capped  events_per_repo  diversity_sco

SPLIT AND SAVE PREPROCESSED DATA

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

# Save datasets
os.makedirs('preprocessed_data', exist_ok=True)

X_train.to_csv('preprocessed_data/X_train.csv', index=False)
X_test.to_csv('preprocessed_data/X_test.csv', index=False)
y_train.to_csv('preprocessed_data/y_train.csv', index=False, header=True)
y_test.to_csv('preprocessed_data/y_test.csv', index=False, header=True)
data.to_csv('preprocessed_data/full_preprocessed_data.csv', index=False)

Train size: 14150 (80.0%)
Test size: 3538 (20.0%)

Train class distribution:
is_active_next_hour
0    10344
1     3806
Name: count, dtype: int64

Test class distribution:
is_active_next_hour
0    2586
1     952
Name: count, dtype: int64


No missing values were found. Outliers were capped at the 99th percentile - total events capped at 66.8 events per hour, unique repositories capped at 6 repos.

Four derived features were created including log transformation which reduced skewness from 5.43 to 1.72. StandardScaler was applied successfully.

The data was split 80-20 with class distribution preserved - 70.5 percent inactive, 29.5 percent active. All files are saved and ready for model training.

## Feature Description for ML Developer

**Features (X):**

- total_events_h12_capped - Number of user events in Hour 12, capped at 99th percentile (max 66.8). Primary activity indicator.

- event_type_diversity_h12 - Number of unique event types performed by user in Hour 12 (range 1 to 10). Shows engagement breadth.

- unique_repos_h12_capped - Number of unique repositories user interacted with in Hour 12, capped at 99th percentile (max 6). Shows GitHub reach.

- events_per_repo - Total events divided by unique repos. Measures activity intensity vs breadth. Higher values mean deeper engagement with fewer repos.

- diversity_score - Event type diversity divided by total events. High score with low total events indicates engaged new user.

- log_total_events - Log transformation of capped total events. Reduces skew for linear models.

- is_bot_suspect - Binary flag for total events above 1000. Currently all zeros due to capping. Can be removed.
- minutes_since_last_action_h12 - Recency feature. Calculates how many minutes before the end of Hour 12 the user performed their last action. Lower values (recent activity) strongly correlate with continued activity in Hour 13.
- push_count_h12 - Number of PushEvents by the user. Pushing code typically indicates a sustained, active working session rather than a brief visit.
- pr_count_h12 - Number of PullRequestEvents by the user. Indicates high-effort collaboration or code review activities.

**Target (y):**

- is_active_next_hour - Binary classification (1 = active, 0 = inactive). User is active in Hour 13 if they had more than 3 events (75th percentile threshold).